In [31]:
!pip install -q google-generativeai pypdf numpy

In [32]:
import google.generativeai as genai
import pypdf
import numpy as np
import os
import textwrap
from google.colab import userdata, files

# Get API key from Colab Secrets
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except KeyError:
    print("GEMINI_API_KEY not found in Colab Secrets.")
    print("Please go to 'Secrets' (key icon on the left) and add your API key.")
    raise

genai.configure(api_key=os.environ['GEMINI_API_KEY'])
print("Successfully configured Gemini API.")

Successfully configured Gemini API.


In [ ]:
# System instruction for the bike rental assistant
SYSTEM_INSTRUCTION = """
You are a helpful bike rental assistant. Your task is to answer user questions based *only* on the provided context.
- Do not use any external knowledge.
- If the answer is not found in the context, you *must* state: "Sorry, I don't have that information in my knowledge base."
- Be concise, polite, and directly answer the question using only the information from the context.
"""

# Generation model (choose from your available list)
GENERATION_MODEL = 'models/gemini-2.5-flash'   # or 'models/gemini-2.5-pro'

# Embedding model (choose from your available list)
EMBEDDING_MODEL = 'models/gemini-embedding-001'   # or 'models/gemini-embedding-2-preview'

generation_model = genai.GenerativeModel(
    model_name=GENERATION_MODEL,
    system_instruction=SYSTEM_INSTRUCTION
)

print("Models initialized.")

Models initialized.


In [34]:
print("Please upload your bike rental information PDF.")
uploaded_files = files.upload()
if not uploaded_files:
    print("\nNo file uploaded. Please run the cell again and select a file.")
else:
    pdf_file_name = list(uploaded_files.keys())[0]
    print(f"\nSuccessfully uploaded: {pdf_file_name}")

Please upload your bike rental information PDF.


Saving Ride_Lanka_Chatbot_Knowledge_Base.pdf to Ride_Lanka_Chatbot_Knowledge_Base (4).pdf

Successfully uploaded: Ride_Lanka_Chatbot_Knowledge_Base (4).pdf


In [35]:
def extract_and_chunk_pdf(file_name):
    print(f"Reading {file_name}...")
    pdf_reader = pypdf.PdfReader(file_name)
    full_text = ""
    for page in pdf_reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n\n"
    print(f"Extracted text length: {len(full_text)} characters.")
    print("First 500 characters of extracted text:")
    print(full_text[:500])
    print("-" * 50)

    # Split into chunks by double newline (paragraphs)
    chunks = full_text.split("\n\n")
    # Remove very short chunks (less than 20 characters)
    chunks = [chunk.strip() for chunk in chunks if len(chunk.strip()) > 20]
    print(f"Created {len(chunks)} text chunks.")
    return chunks, full_text

# Run the function
text_chunks, full_text = extract_and_chunk_pdf(pdf_file_name)

# Show some example chunks
print("\n--- Example Chunks ---")
for i, chunk in enumerate(text_chunks[:3]):
    print(f"Chunk {i+1} (first 200 chars):")
    print(textwrap.fill(chunk[:200], width=80))
    print("---")

Reading Ride_Lanka_Chatbot_Knowledge_Base (4).pdf...
Extracted text length: 17045 characters.
First 500 characters of extracted text:
Ride Lanka - AI Chatbot Knowledge 
Base 
1. About Sri Lanka 
Sri Lanka is an island nation in the Indian Ocean known for its rich cultural heritage, scenic 
landscapes, and tourism attractions. Popular destinations include Ella, Galle, Mirissa, 
Hikkaduwa, and Kandy. The country offers beaches, mountains, wildlife, and historical sites. 
2. Tourism in Sri Lanka 
Sri Lanka attracts millions of tourists annually due to its biodiversity, UNESCO heritage 
sites, and cultural festivals. Tourism contr
--------------------------------------------------
Created 10 text chunks.

--- Example Chunks ---
Chunk 1 (first 200 chars):
Ride Lanka - AI Chatbot Knowledge  Base  1. About Sri Lanka  Sri Lanka is an
island nation in the Indian Ocean known for its rich cultural heritage, scenic
landscapes, and tourism attractions. Popula
---
Chunk 2 (first 200 chars):
Q: Can I

In [36]:
def embed_text(text):
    """Embed a single piece of text using the chosen embedding model."""
    try:
        result = genai.embed_content(
            model=EMBEDDING_MODEL,
            content=text,
            task_type="RETRIEVAL_DOCUMENT"
        )
        return result['embedding']
    except Exception as e:
        print(f"Error embedding text: {e}")
        print(f"Text snippet: {text[:100]}...")
        return None

# Test embedding with first chunk
if text_chunks:
    test_embedding = embed_text(text_chunks[0])
    if test_embedding is not None:
        print(f"Test embedding successful. Vector length: {len(test_embedding)}")
    else:
        print("Test embedding failed. Check model name and API key.")
else:
    print("No text chunks available. Cannot proceed.")

# Build knowledge base
knowledge_base = []
print("\nEmbedding all chunks (this may take a minute)...")
for i, chunk in enumerate(text_chunks):
    if (i+1) % 10 == 0:
        print(f"Embedding chunk {i+1}/{len(text_chunks)}...")
    embedding = embed_text(chunk)
    if embedding is not None:
        knowledge_base.append({
            'text': chunk,
            'embedding': np.array(embedding)
        })

print(f"\nEmbedding complete. Knowledge base has {len(knowledge_base)} embedded chunks.")
if knowledge_base:
    print("First chunk embedding shape:", knowledge_base[0]['embedding'].shape)
else:
    print("WARNING: knowledge_base is empty! No embeddings were created.")

Test embedding successful. Vector length: 3072

Embedding all chunks (this may take a minute)...
Embedding chunk 10/10...

Embedding complete. Knowledge base has 10 embedded chunks.
First chunk embedding shape: (3072,)


In [37]:
def embed_query(query):
    """Embed the user's question."""
    try:
        result = genai.embed_content(
            model=EMBEDDING_MODEL,
            content=query,
            task_type="RETRIEVAL_QUERY"
        )
        return result['embedding']
    except Exception as e:
        print(f"Error embedding query: {e}")
        return None

def find_most_similar_chunks(query, top_k=3):
    """Return the top_k most similar text chunks for the given query."""
    if not knowledge_base:
        return ""
    query_vec = embed_query(query)
    if query_vec is None:
        return "Error embedding query."
    query_vec = np.array(query_vec)

    similarities = []
    for item in knowledge_base:
        dot = np.dot(query_vec, item['embedding'])
        norm_q = np.linalg.norm(query_vec)
        norm_i = np.linalg.norm(item['embedding'])
        if norm_q and norm_i:
            sim = dot / (norm_q * norm_i)
        else:
            sim = 0.0
        similarities.append((sim, item['text']))

    similarities.sort(reverse=True, key=lambda x: x[0])
    # Join the top_k texts
    context = "\n---\n".join([text for _, text in similarities[:top_k]])
    return context

# Test retrieval with a sample question
test_query = "What are popular destinations in Sri Lanka?"
context = find_most_similar_chunks(test_query, top_k=3)
print("Test retrieval result:")
if context:
    print(textwrap.fill(context[:500], width=80))
else:
    print("No context retrieved. Check knowledge_base and embedding.")

Test retrieval result:
Q15: Can I modify my booking?  A: Yes, before confirmation.  Q16: How is rental
cost calculated?  A: Days × price × quantity.      5. Vendor Questions  Q17: How
do I become a vendor?  A: Register as vendor and submit details.  Q18: How are
bikes approved?  A: Admin reviews and approves listings.  Q19: Can vendors
update bike details?  A: Yes, anytime.      6. Tourism Questions  Q20: What are
the best places in Sri Lanka?  A: Ella, Galle, Mirissa, Hikkaduwa, Kandy.  Q21:
What is Ella famous for?


In [43]:
def get_chat_response(query, context):
    """Generate an answer using the generation model."""
    final_prompt = f"""
Context:
---
{context}
---
Question:
{query}
Answer:
"""
    try:
        response = generation_model.generate_content(
            final_prompt,
            generation_config=genai.types.GenerationConfig(temperature=0.1)
        )
        return response.text
    except Exception as e:
        print(f"Error generating chat response: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Optional: test generation with the retrieved context
if context:
    test_answer = get_chat_response(test_query, context)
    print("\n--- Test Answer ---")
    print(test_answer)
    print("-------------------")

Error generating chat response: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 47.22696394s.

--- Test Answer ---
Sorry, I encountered an error while generating the answer.
-------------------


In [39]:
def ask_question(question):
    print("Retrieving relevant context...")
    context = find_most_similar_chunks(question, top_k=3)
    if not context:
        print("No relevant context found.")
        return
    print("\n--- Found Context ---")
    print(textwrap.fill(context, width=80))
    print("---------------------\n")
    print("Generating answer...")
    answer = get_chat_response(question, context)
    print("\n--- Answer ---")
    print(answer)
    print("--------------")

# Example question - change to suit your PDF
user_question = "What types of bikes do you have for rent and what are the rental prices?"
ask_question(user_question)

Retrieving relevant context...

--- Found Context ---
to LKR 3,500. For larger, geared motorbikes (such as a Honda XR 150 or Royal
Enfield) that  are better suited for hilly terrains or long-distance touring,
the daily rate ranges from LKR  4,500 to LKR 7,000. Our platform uses an AI-
driven dynamic pricing model to ensure these  rates are always fair, preventing
tourists from being overcharged during peak seasons while  optimizing revenue
for vendors.    Part 4: Q&A Pairs (For Intent and Feature Training)  Q: How much
does it cost to rent a scooter for one day in the South?A: A standard  scooter
typically costs between LKR 2,000 and LKR 3,500 per day. However, our system
uses dynamic pricing, so the exact rate might adjust slightly based on current
demand and  the season to guarantee you get a fair deal.  Q: Do I get a discount
if I rent the bike for a whole week?A: Yes! Vendors on our  platform can set
specific rates for longer durations. If you book a bike for a week or a  month,
th

In [42]:
print("Starting interactive chat. Type 'quit' to exit.")
print("-" * 30)
while True:
    user_question = input("Ask a question: ")
    if user_question.lower() == 'quit':
        print("Goodbye!")
        break
    ask_question(user_question)
    print("\n")

Starting interactive chat. Type 'quit' to exit.
------------------------------
Ask a question: what is sri lanka
Retrieving relevant context...

--- Found Context ---
Ride Lanka - AI Chatbot Knowledge  Base  1. About Sri Lanka  Sri Lanka is an
island nation in the Indian Ocean known for its rich cultural heritage, scenic
landscapes, and tourism attractions. Popular destinations include Ella, Galle,
Mirissa,  Hikkaduwa, and Kandy. The country offers beaches, mountains, wildlife,
and historical sites.  2. Tourism in Sri Lanka  Sri Lanka attracts millions of
tourists annually due to its biodiversity, UNESCO heritage  sites, and cultural
festivals. Tourism contributes significantly to the economy and supports  local
businesses.  3. About Ride Lanka Platform  Ride Lanka is a web-based bike rental
platform that connects tourists with local bike  vendors. Users can browse
bikes, book rentals, and explore tourist destinations. Vendors  can list bikes,
and admins approve listings.  4. Bike Rent

Error generating chat response: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 37.780772991s.

--- Answer ---
Sorry, I encountered an error while generating the answer.
--------------


Ask a question: what is sri lanka
Retrieving relevant context...

--- Found Context ---
Ride Lanka - AI Chatbot Knowledge  Base  1. About Sri Lanka  Sri Lanka is an
island nation in the Indian Ocean known for its rich cultural heritage, scenic
landscapes, and tourism attractions. Popular destinations include Ella, Galle,
Mirissa,  Hik